In [1]:
import sys; sys.path.append("../src")
import pandas as pd
from evaluate import load_truth, split_ids, f05_one, f05_score, write_tsv

# Load the answer key
gt = pd.read_pickle("../work/tr_gt.pkl")
truth = load_truth(gt)

# Split into practice-train (80%) and practice-exam (20%)
tr_ids, val_ids = split_ids(truth.keys())
print("Train S1:", len(tr_ids), "| Validation S1:", len(val_ids))

# Check 1: example from the PDF should give 0.714
print("PDF example:", round(f05_one(["S2-00047", "S2-00193", "S3-00812"], ["S2-00047", "S3-00812"]), 3))

# Check 2: a perfect answer should give 1.0
print("Perfect:", f05_score(truth, truth, val_ids))

# Check 3: saying 'no match' for everyone (worst possible baseline)
print("All-empty baseline:", round(f05_score({}, truth, val_ids), 4))

# Check 4: write dummy test files to check the format
te_s1 = pd.read_pickle("../work/te_s1.pkl")
write_tsv({}, te_s1.entity_id, "../output/matching_results.tsv")
write_tsv({}, te_s1.entity_id, "../output/candidate_pairs.tsv", col="candidate_entity_ids")
print("Dummy files written")

Train S1: 1765621 | Validation S1: 441200
PDF example: 0.714
Perfect: 1.0
All-empty baseline: 0.0558
Dummy files written


In [2]:
from evaluate import pairs_to_pred, tune_threshold
import numpy as np

# Fake model output to test the tool:
# true pairs get random scores 0.4-1.0, wrong pairs get 0.0-0.7
val_list = list(val_ids)[:20000]
rng = np.random.default_rng(0)
rows = []
for s1 in val_list:
    for m in truth[s1]:
        rows.append((s1, m, rng.uniform(0.4, 1.0)))            # true pair
    rows.append((s1, f"S2-FAKE{s1}", rng.uniform(0.0, 0.7)))    # wrong pair
fake = pd.DataFrame(rows, columns=["s1_id", "other_id", "prob"])

tune_threshold(fake, truth, val_list)

threshold 0.30 -> F0.5 0.8568
threshold 0.35 -> F0.5 0.8750
threshold 0.40 -> F0.5 0.8922
threshold 0.45 -> F0.5 0.8828
threshold 0.50 -> F0.5 0.8699
threshold 0.55 -> F0.5 0.8517
threshold 0.60 -> F0.5 0.8268
threshold 0.65 -> F0.5 0.7908
threshold 0.70 -> F0.5 0.7455
threshold 0.75 -> F0.5 0.6734
threshold 0.80 -> F0.5 0.5866
threshold 0.85 -> F0.5 0.4835
threshold 0.90 -> F0.5 0.3622
threshold 0.95 -> F0.5 0.2195
BEST: 0.4 0.8922


(0.4, 0.8921534062060459)